# Stage 1a — rate CFD with an open-weight VLMRuns the behavioral half of the audit: every neutral CFD image scored by onemodel, zero-shot, using the ordinal expected-rating task score.**Before you start**1. Runtime → Change runtime type → **L4 GPU** (Colab Pro). The free T4 has 16 GB   and a 7B VLM at fp16 needs ~17 GB — see the guard cell below.2. Put `CFD.zip` (the whole `CFD Version 3.0` folder, zipped) in your Drive.3. Set `REPO_URL` below to your private GitHub repo.**Do not quantize to fit a smaller GPU.** The dependent variable is the logitdistribution over the seven rating tokens; quantization perturbs exactly that,and at Stage 4 it contaminates the gradients the CAV sensitivity is built from.

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csvimport torchassert torch.cuda.is_available(), "No GPU. Runtime -> Change runtime type -> GPU."name = torch.cuda.get_device_name(0)vram = torch.cuda.get_device_properties(0).total_memory / 1e9print(f"\n{name}  |  {vram:.1f} GB")if vram < 20:    raise RuntimeError(        f"{name} has only {vram:.1f} GB. A 7B VLM at fp16 needs ~17 GB of weights "        "alone. Switch to an L4 (24 GB) or A100. Do not work around this by "        "quantizing -- see the note at the top."    )

## 2. Get the code

In [ ]:
REPO_URL = "https://github.com/YOURUSER/face.git"  # <-- set thisimport os, subprocessif not os.path.exists("/content/face"):    subprocess.run(["git", "clone", REPO_URL, "/content/face"], check=True)os.chdir("/content/face")!git log --oneline -1

## 3. Install dependencies\n\nTorch is intentionally not reinstalled -- Colab's build is CUDA-matched.

In [ ]:
!pip install -q -r requirements.txt

## 4. Stage the CFD images on local diskRead images from `/content`, never from mounted Drive. 831 large JPEGs overDrive's FUSE layer is dramatically slower than unzipping once.

In [ ]:
from google.colab import drivedrive.mount("/content/drive")CFD_ZIP = "/content/drive/MyDrive/CFD.zip"  # <-- adjust if neededimport pathlibif not pathlib.Path("/content/face/dataset").exists():    !mkdir -p /content/face/dataset && unzip -q "$CFD_ZIP" -d /content/face/dataset!ls "/content/face/dataset"

## 5. Verify the pipeline before spending GPU timeRuns the full test suite (no GPU, no model needed) and builds the manifest.Expect 25 passing tests and 831 rows / 826 matched. If this fails, stop here --nothing downstream can be right.

In [ ]:
!python -m pytest tests/ -qfrom pathlib import Pathfrom facecav.data.cfd import build_manifestmanifest = build_manifest(Path("dataset/CFD Version 3.0"))print(f"\nrows: {len(manifest)}   matched: {(manifest.join_status == 'matched').sum()}")print(manifest.join_status.value_counts().to_string())

## 6. Smoke test — three imagesThe first real forward pass. This is where the untested assumptions in`rater.py` surface: whether the chat template renders, whether the processoraccepts the image, and whether the logits land on the position after`"The rating is "`.**Sanity checks on the output:** `expected_rating` must sit inside [1, 7] andmust not be identical across three different faces (that would mean the imageis being ignored). `refusal_mass` near 1.0 means the model is not answeringwith a digit at all.

In [ ]:
MODEL = "Qwen/Qwen2.5-VL-7B-Instruct"!python experiments/stage1a_rate_cfd.py --model "$MODEL" --limit 3import json, pathlibout = pathlib.Path("artifacts/stage1a") / f"{MODEL.replace('/', '__')}.jsonl"for line in out.read_text().splitlines():    r = json.loads(line)    print(f"{r['model_id']:>12}  rating={r['expected_rating']:.3f}  "          f"refusal={r['refusal_mass']:.4f}  probs={[round(p, 3) for p in r['rating_probs']]}")

## 7. Full Stage 1aAll 831 images. Resumable: results append to JSONL and completed images areskipped, so a disconnect costs at most one image -- just rerun this cell.

In [ ]:
!python experiments/stage1a_rate_cfd.py --model "$MODEL"

## 8. First look at the results

In [ ]:
import pandas as pdratings = pd.read_json(out, lines=True)print(f"n = {len(ratings)}")print(f"\nrefusal mass: mean={ratings.refusal_mass.mean():.4f}  max={ratings.refusal_mass.max():.4f}")print("\nmean expected rating by race x gender:")print(pd.crosstab(ratings.race_code, ratings.gender_code,                  values=ratings.expected_rating, aggfunc="mean").round(3).to_string())print("\nrefusal by race (differential refusal is itself a finding -- spec 5.6):")print(ratings.groupby("race_code").refusal_mass.mean().round(4).to_string())

## 9. Save results back to DriveColab storage is ephemeral. The JSONL is small; always copy it out.

In [ ]:
!mkdir -p /content/drive/MyDrive/face_artifacts!cp -r artifacts/stage1a /content/drive/MyDrive/face_artifacts/!ls -la /content/drive/MyDrive/face_artifacts/stage1a

---### NextRepeat section 7 for the other models (`OpenGVLab/InternVL3-8B`,`HuggingFaceM4/Idefics3-8B-Llama3`), then the ICL condition. The 32B/38B armneeds an A100 80GB — set the High-RAM toggle.Numbers here are **not** a result yet. Before anything gets reported, check theprompt-paraphrase stability and refusal rates called for in spec section 9.